# Chapitre 11 - Exercices 11.5 à 11.7

Ces trois exercices sont des travaux pratiques : programmer un réseau de neurones avec
rétropropagation et weight decay et l'étudier sur une simulation (11.5), programmer la
projection pursuit regression et la comparer au réseau (11.6), puis comparer réseau et modèle
additif sur les données *spam* (11.7).

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from esl.plotting import set_style, COLORS, BLUE, ORANGE, GRAY, SEQ, DIV, TWO_CLASS
set_style()

from esl import datasets
from esl.ch11_reseaux import MLP, PPR, SmoothingSpline, backprop, one_hot, sigmoid

## Exercice 11.5 : Réseau à dix unités : époques, weight decay et nombre d'unités

**Énoncé (reformulé).**

(a) Écrire un programme qui ajuste un réseau à une couche cachée (dix unités) par
rétropropagation avec weight decay.

(b) L'appliquer à 100 observations du modèle

$$Y = \sigma(a_1^TX) + (a_2^TX)^2 + 0{,}30\cdot Z,$$

où $\sigma$ est la sigmoïde, $Z$ une gaussienne centrée réduite, $X = (X_1, X_2)$ à composantes
gaussiennes centrées réduites indépendantes, $a_1 = (3, 3)$ et $a_2 = (3, -3)$. Tirer un
échantillon de test de taille 1000 et tracer les erreurs d'apprentissage et de test en
fonction du nombre d'époques, pour plusieurs valeurs du paramètre de weight decay. Discuter le
sur-ajustement dans chaque cas.

(c) Faire varier le nombre d'unités cachées de 1 à 10 et déterminer le nombre minimal
nécessaire pour bien résoudre ce problème.

### (a) Le programme

Le programme est la classe `MLP` du module `esl/ch11_reseaux.py`, dont le cœur est la fonction
`backprop` (cours 2) :

* passe avant : $z_{mi} = \sigma(\alpha_{0m} + \alpha_m^Tx_i)$, $f(x_i) = \beta_0 + \beta^Tz_i$ ;
* passe arrière : $\delta_i = -2(y_i - f(x_i))$, $s_{mi} = z_{mi}(1 - z_{mi})\,\beta_m\delta_i$ ;
* gradient : $\partial R/\partial\beta_m = \sum_i\delta_i z_{mi} + 2\lambda\beta_m$,
  $\partial R/\partial\alpha_{m\ell} = \sum_i s_{mi}x_{i\ell} + 2\lambda\alpha_{m\ell}$ (constantes non pénalisées).

Pour l'entraînement par époques, nous écrivons ici explicitement la boucle de **descente de
gradient par lots avec inertie** (*momentum*) : on garde une « vitesse » $v$ et l'on itère

$$v \leftarrow \mu\, v - \gamma\,\frac{1}{N}\nabla\big(R + \lambda J\big)(\theta), \qquad \theta \leftarrow \theta + v,$$

avec $\mu \in [0, 1)$. Pour $\mu = 0$ c'est la descente (11.13) ; $\mu > 0$ accumule les directions
de descente persistantes et accélère beaucoup la traversée des plateaux.